# Solver Segment Efficiency — Fixed Acceptance (ε = 2 mrad)

New-data remake of the classical/quantum **solver segment-efficiency** 2×2
figure, reading the decoupled `qtrk_store` **metrics view**.
**Nothing is re-solved here** — the Condor solves are read back and the segment
metrics are the recomputed view at the γ-aware absolute threshold
τ = δ/(δ+γ) + 0.10 (the definition on the Notion source-of-truth page).

**Data:** fixed ε = 2 mrad (`eps_provenance='set'`), σ_scatt = 1e-4, σ_res = 0,
φ_max = 0.2, step kernel.  Sweep: n_trk ∈ {10 … 1000}, γ ∈ {1, 2, 3},
hit inefficiency ∈ {0, 1 %}.

**2×2 panels:**
| Panel | Quantity | Definition |
|---|---|---|
| i)  | segment efficiency | N_true,active / N_true,all |
| ii) | segment false rate | N_false,active / N_active |
| iii)| total segment pairs | N_true,all and N_false,all vs n_trk |
| iv) | active segment pairs | N_true,active and N_false,active vs n_trk |

**Headline:** γ = 3, δ = 1, τ = 0.35 — the original operating point — for
**classical** and **quantum**, at the 1 % hit-drop working point.
**Companion:** γ ∈ {1, 2, 3} overlay of the two key panels.

In [1]:
import sys
sys.path.insert(0, "/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Segment_level_studies")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seg_store as S

plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                      "axes.grid": True, "grid.alpha": 0.3})
print("fixed acceptance ε =", S.EPS_FIXED, "rad (2 mrad)")
print("thresholds:  γ=1 -> τ=%.3f   γ=2 -> τ=%.3f   γ=3 -> τ=%.3f"
      % (S.threshold(1), S.threshold(2), S.threshold(3)))

OUT = Path(S.__file__).resolve().parent / "outputs" / "solver_segment_efficiency"
OUT.mkdir(parents=True, exist_ok=True)

fixed acceptance ε = 0.002 rad (2 mrad)
thresholds:  γ=1 -> τ=0.600   γ=2 -> τ=0.433   γ=3 -> τ=0.350


In [2]:
# ---- load the recomputed metrics view (fixed-ε only) ----------------------
M = S.fixed_eps_metrics()
READY = len(M) > 0
if not READY:
    print("[FLAG] metrics view has no fixed-ε (ε=2 mrad) rows yet.")
    print("       build_metrics.py is still running; it writes metrics.csv at the end.")
    print("       Re-run this notebook once it lands — every figure below will fill in.")
else:
    print(f"fixed-ε metrics rows: {len(M)}")
    inv = (M.groupby(["solver", "gamma", "hit_ineff"])["n_trk"]
             .agg(points="nunique", solves="count"))
    print(inv.to_string())

fixed-ε metrics rows: 1068
                           points  solves
solver    gamma hit_ineff                
classical 1.0   0.00            8     160
                0.01            8     160
          2.0   0.00            8     160
                0.01            8     160
          3.0   0.00            8     160
                0.01            8     160
quantum   1.0   0.00            6      18
                0.01            6      18
          2.0   0.00            6      18
                0.01            6      18
          3.0   0.00            6      18
                0.01            6      18


In [3]:
# ---- the shared 2×2 plotter (matches the original §-style figure) ---------
C_EFF, C_FR = "#1b7837", "#c51b7d"
C_TRUE, C_FALSE = "#2166ac", "#d6604d"

def plot_2x2(d, title, stem):
    """d = S.agg_by_ntrk(...) dict; draws + saves the 2×2 figure."""
    if not d:
        print(f"[FLAG] no data for: {title}  — skipped (config absent in store).")
        return
    tc = d["tc"]
    fig, ax = plt.subplots(2, 2, figsize=(12, 10))

    a = ax[0, 0]
    a.errorbar(tc, d["se_m"], yerr=d["se_e"], fmt="o-", color=C_FR,
               capsize=4, mec="k", mew=0.7, zorder=3)
    a.axhline(100, color="gray", ls="--", lw=1.0, alpha=0.7)
    lo = max(0, np.floor((d["se_m"] - d["se_e"]).min()) - 1)
    a.set_ylim(lo, 100.8)
    a.set_ylabel("Segment efficiency (%)")
    a.set_title(r"i) Segment efficiency $N_{\rm true\,act}/N_{\rm true\,all}$",
                fontweight="bold")
    a.yaxis.set_major_formatter(mticker.FormatStrFormatter("%g%%"))

    a = ax[0, 1]
    a.errorbar(tc, d["fr_m"], yerr=d["fr_e"], fmt="s-", color=C_FR,
               capsize=4, mec="k", mew=0.7, zorder=3)
    a.set_ylabel("Segment false rate (%)")
    a.set_title(r"ii) Segment false rate $N_{\rm false\,act}/N_{\rm active}$",
                fontweight="bold")
    a.yaxis.set_major_formatter(mticker.FormatStrFormatter("%g%%"))

    a = ax[1, 0]
    a.errorbar(tc, d["ntm"], yerr=d["ntse"], fmt="o-", color=C_TRUE,
               capsize=3, mec="k", mew=0.7, label="True segments", zorder=3)
    a.errorbar(tc, d["nfm"], yerr=d["nfse"], fmt="s-", color=C_FALSE,
               capsize=3, mec="k", mew=0.7, label="False segments", zorder=3)
    a.set_yscale("log"); a.set_ylabel("Number of segment pairs")
    a.set_title("iii) Segment-pair counts", fontweight="bold")
    a.legend(loc="upper left", fontsize=10)

    a = ax[1, 1]
    fa = np.where(d["fam"] > 0, d["fam"], 0.5)
    fae = np.where(d["fam"] > 0, d["fase"], 0.0)
    a.errorbar(tc, d["tam"], yerr=d["tase"], fmt="o-", color=C_TRUE,
               capsize=3, mec="k", mew=0.7, label="True active", zorder=3)
    a.errorbar(tc, fa, yerr=fae, fmt="s-", color=C_FALSE,
               capsize=3, mec="k", mew=0.7, label="False active", zorder=3)
    a.set_yscale("log"); a.set_ylim(0.1, None)
    a.set_ylabel("Number of active segments")
    a.set_title("iv) Active segment pairs", fontweight="bold")
    a.legend(loc="lower right", fontsize=10)

    for x in ax.flat:
        x.set_xlabel("Number of tracks")
        x.tick_params(which="both", direction="in", top=True, right=True)
    for x in ax[1, :]:
        x.set_xscale("log")
    fig.suptitle(title, fontsize=13, fontweight="bold", y=1.00)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    for ext, dpi in (("pdf", 600), ("png", 300)):
        fig.savefig(OUT / f"{stem}.{ext}", dpi=dpi, bbox_inches="tight",
                    facecolor="white")
    plt.show()
    print("saved", stem, "(reps/point >=", d.get("n_reps"), ")")

## Headline — γ = 3 (the original operating point)

In [4]:
if READY:
    plot_2x2(S.agg_by_ntrk(M, "classical", gamma=3.0, hit_ineff=0.01),
             r"Classical solver — segment metrics (ε=2 mrad, γ=3, 1% hit drop)",
             "classical_2x2_g3_drop1pct")
    plot_2x2(S.agg_by_ntrk(M, "classical", gamma=3.0, hit_ineff=0.0),
             r"Classical solver — segment metrics (ε=2 mrad, γ=3, no hit drop)",
             "classical_2x2_g3_base")

saved classical_2x2_g3_drop1pct (reps/point >= 20 )


saved classical_2x2_g3_base (reps/point >= 20 )


In [5]:
if READY:
    plot_2x2(S.agg_by_ntrk(M, "quantum", gamma=3.0, hit_ineff=0.01),
             r"Quantum (1BQF) solver — segment metrics (ε=2 mrad, γ=3, 1% hit drop)",
             "quantum_2x2_g3_drop1pct")
    plot_2x2(S.agg_by_ntrk(M, "quantum", gamma=3.0, hit_ineff=0.0),
             r"Quantum (1BQF) solver — segment metrics (ε=2 mrad, γ=3, no hit drop)",
             "quantum_2x2_g3_base")
    print("NOTE: quantum reps are few (3 at low n, 1 at n>=700) -> wide/zero error bars at high n.")

saved quantum_2x2_g3_drop1pct (reps/point >= 3 )


saved quantum_2x2_g3_base (reps/point >= 3 )
NOTE: quantum reps are few (3 at low n, 1 at n>=700) -> wide/zero error bars at high n.


## Companion — γ ∈ {1, 2, 3} overlay (efficiency & false rate, 1 % drop)

In [6]:
if READY:
    GCOL = {1.0: "#4575b4", 2.0: "#984ea3", 3.0: "#d73027"}
    fig, ax = plt.subplots(2, 2, figsize=(13, 10))
    for si, solver in enumerate(["classical", "quantum"]):
        for g in (1.0, 2.0, 3.0):
            d = S.agg_by_ntrk(M, solver, gamma=g, hit_ineff=0.01)
            if not d:
                continue
            ax[si, 0].errorbar(d["tc"], d["se_m"], yerr=d["se_e"], fmt="o-",
                               color=GCOL[g], capsize=3, mec="k", mew=0.5,
                               label=fr"$\gamma={g:g}$ ($\tau$={S.threshold(g):.2f})")
            ax[si, 1].errorbar(d["tc"], d["fr_m"], yerr=d["fr_e"], fmt="s-",
                               color=GCOL[g], capsize=3, mec="k", mew=0.5,
                               label=fr"$\gamma={g:g}$")
        ax[si, 0].set_ylabel(f"{solver}\nSegment efficiency (%)")
        ax[si, 1].set_ylabel("Segment false rate (%)")
        for j in (0, 1):
            ax[si, j].set_xscale("log"); ax[si, j].set_xlabel("Number of tracks")
            ax[si, j].legend(fontsize=9)
            ax[si, j].tick_params(which="both", direction="in", top=True, right=True)
    ax[0, 0].set_title("Segment efficiency", fontweight="bold")
    ax[0, 1].set_title("Segment false rate", fontweight="bold")
    fig.suptitle(r"γ sweep — fixed ε = 2 mrad, 1% hit drop (top: classical, bottom: quantum)",
                 fontsize=13, fontweight="bold", y=1.00)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    for ext, dpi in (("pdf", 600), ("png", 300)):
        fig.savefig(OUT / f"gamma_sweep_companion.{ext}", dpi=dpi,
                    bbox_inches="tight", facecolor="white")
    plt.show()
    print("saved gamma_sweep_companion")

saved gamma_sweep_companion


## Summary table (CSV export)

In [7]:
if READY:
    rows = []
    for solver in ["classical", "quantum"]:
        for g in (1.0, 2.0, 3.0):
            for hi in (0.0, 0.01):
                d = S.agg_by_ntrk(M, solver, gamma=g, hit_ineff=hi)
                if not d:
                    continue
                for i, n in enumerate(d["tc"]):
                    rows.append(dict(solver=solver, gamma=g, hit_ineff=hi,
                                     n_trk=int(n), eff_pct=round(d["se_m"][i], 3),
                                     eff_sem=round(d["se_e"][i], 3),
                                     far_pct=round(d["fr_m"][i], 3),
                                     far_sem=round(d["fr_e"][i], 3),
                                     n_true=round(d["ntm"][i], 1),
                                     n_false_all=round(d["nfm"][i], 1)))
    tab = pd.DataFrame(rows)
    tab.to_csv(OUT / "segment_efficiency_summary.csv", index=False)
    print("wrote segment_efficiency_summary.csv  (", len(tab), "rows )")
    display(tab[(tab.solver=="classical") & (tab.gamma==3.0) & (tab.hit_ineff==0.01)])

wrote segment_efficiency_summary.csv  ( 84 rows )


,solver,gamma,hit_ineff,n_trk,eff_pct,eff_sem,far_pct,far_sem,n_true,n_false_all
40,classical,3.0,0.01,10,97.895,0.592,0.000,0.000,39.1,351.9
41,classical,3.0,0.01,20,97.887,0.547,0.000,0.000,78.1,1484.2
42,classical,3.0,0.01,50,98.635,0.273,0.205,0.161,196.4,9622.0
43,classical,3.0,0.01,100,98.186,0.190,0.283,0.119,390.4,38645.5
44,classical,3.0,0.01,200,98.571,0.122,0.964,0.175,785.2,156259.0
45,classical,3.0,0.01,400,98.416,0.081,2.579,0.244,1567.0,625261.8
46,classical,3.0,0.01,700,98.458,0.096,8.778,0.395,2742.8,1917287.2
47,classical,3.0,0.01,1000,98.544,0.060,20.827,0.440,3920.2,3916431.8
